In [4]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()

while not (PROJECT_ROOT / "ai_engine").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Could not find project root.")
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: C:\Users\nilarath\OneDrive - Capgemini\Documents\Loan-Application-Tracker


In [5]:

from ai_engine.llm.gateway import LLMGateway
    
# Initialize the central gateway
gateway = LLMGateway()

# Test a normal LLM interaction
response = gateway.invoke_text(
    "You are an AI assistant for a loan application tracking system. "
    "Explain your role in exactly two sentences."
)

print(response)


I help you track and manage loan applications by providing real-time status updates, required document checklists, deadline reminders, and clear explanations of each step in the process. I can answer questions about your application, flag missing items, connect you to support or lender contacts, and ensure your sensitive information is handled according to privacy and security policies.


In [6]:
from pydantic import BaseModel

class LoanAssistantResponse(BaseModel):
    answer: str
    category: str
    requires_human_review: bool

result = gateway.invoke_structured(
    """
    An applicant asks what documents are required
    for a salaried personal loan.
    """,
    LoanAssistantResponse
)

print(result)
print(type(result))

answer='Typical documents required for a salaried personal loan (may vary by lender and country):\n\n- Proof of identity: passport, driver’s license, national ID card or Aadhaar\n- Proof of address: utility bill, rental agreement, passport, driver’s license or Aadhaar\n- PAN or tax ID (India) or equivalent tax identification document in your country\n- Salary proof: last 2–3 months’ salary slips (some lenders require last 6 months)\n- Bank statements: last 3–6 months’ bank statements showing salary credits\n- Employment proof: employment/appointment letter, or employer confirmation and employee ID\n- Income tax documents: Form 16 or last 1–2 years’ ITRs (may be required for higher amounts)\n- Recent passport-size photograph(s)\n- Cancelled cheque or bank passbook copy for account details / ECS mandate\n- Completed loan application form and signed declarations\n- Details of existing loans/EMIs (if any) and supporting statements\n\nTips: lenders’ checklists vary—verify their specific req

In [8]:
from ai_engine.llm.prompts import LOAN_ASSISTANT_PROMPT

# Sample context for testing
context = """
Loan type: Personal Loan
Applicant type: Salaried
Monthly income: INR 50,000
Application status: Documents Pending
"""

question = "What is the current status of my loan application and what is my monthly salary ?"

# Generate messages using our prompt template
messages = LOAN_ASSISTANT_PROMPT.format_messages(
    context=context,
    question=question
)

# Send messages through our central gateway
response = gateway.invoke(messages)

print(response.content)

Current status (from your application): Documents Pending.

Monthly salary (as entered): INR 50,000 per month.

What "Documents Pending" means and next steps:
- It indicates the lender has not yet received or completed verification of all required documents for your application. I don’t have the specific list of which documents are missing.
- To find the exact missing items, please check your loan portal/dashboard, the email/SMS from the lender, or contact your loan officer.
- Common documents for a salaried personal loan (you can prepare these) include: identity proof (PAN, Aadhaar, passport), address proof, last 3 months’ salary slips, last 6 months’ bank statements, employment proof/offer letter, and income tax documents (Form 16) if requested.
- After you submit the missing documents, the lender will verify them. I cannot confirm verification or approval — only the lender or loan officer can.

If you’d like, I can list a more detailed document checklist or help draft a message to y